# Pizza Vision: continuous pipeline simulator

Drops sample camera frames into the `frames_inbox` Unity Catalog volume on
a configurable cadence so the SDP pipeline has something to process.

The pipeline reads the inbox via Auto Loader, dedupes to one frame per
10s per camera, then runs YOLO. The simulator deliberately writes faster
than the dedupe window so most frames are dropped (which is exactly what
we're demonstrating).

Drop layout: `frames_inbox/<camera>/frame_<utc-millis>.jpg`.

In [ ]:
dbutils.widgets.text("catalog", "reggie_pierce_7405614800873570")
dbutils.widgets.text("schema", "pizza_vision")
dbutils.widgets.text("cameras", "cam01,cam02,cam03")
dbutils.widgets.text("frames_per_camera", "60")
dbutils.widgets.text("cadence_seconds", "2")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
CAMERAS = [c.strip() for c in dbutils.widgets.get("cameras").split(",") if c.strip()]
FRAMES_PER_CAMERA = int(dbutils.widgets.get("frames_per_camera"))
CADENCE_S = float(dbutils.widgets.get("cadence_seconds"))
INBOX = f"/Volumes/{CATALOG}/{SCHEMA}/frames_inbox"
print(f"Writing to: {INBOX}")
print(f"Cameras: {CAMERAS}, frames/cam: {FRAMES_PER_CAMERA}, cadence: {CADENCE_S}s")

In [ ]:
import logging
import os
import random
import time
import urllib.request

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("pipeline_simulator")

# Use ultralytics' public sample assets. These exercise the COCO classes the
# YOLO endpoint knows (person, car, bus, pizza, etc.).
_SAMPLE_URLS = [
    "https://github.com/ultralytics/assets/raw/main/im/bus.jpg",
    "https://github.com/ultralytics/assets/raw/main/im/zidane.jpg",
    "https://ultralytics.com/images/bus.jpg",
    "https://ultralytics.com/images/zidane.jpg",
]

_LOCAL_DIR = "/tmp/pizza_vision_samples"
os.makedirs(_LOCAL_DIR, exist_ok=True)

_LOCAL_PATHS: list[str] = []
for url in _SAMPLE_URLS:
    name = os.path.basename(url)
    target = os.path.join(_LOCAL_DIR, name)
    if not os.path.exists(target):
        try:
            urllib.request.urlretrieve(url, target)
            LOG.info("Downloaded %s -> %s", url, target)
        except Exception as exc:
            LOG.warning("Skipping %s (%s)", url, exc)
            continue
    _LOCAL_PATHS.append(target)

if not _LOCAL_PATHS:
    raise RuntimeError("No sample images could be downloaded for the simulator.")
LOG.info("Using %d sample image(s).", len(_LOCAL_PATHS))

In [ ]:
for cam in CAMERAS:
    cam_dir = f"{INBOX}/{cam}"
    try:
        dbutils.fs.mkdirs(cam_dir)
    except Exception as exc:
        LOG.warning("mkdirs failed for %s: %s", cam_dir, exc)
    LOG.info("Camera dir ready: %s", cam_dir)

In [ ]:
rng = random.Random()
total = FRAMES_PER_CAMERA * len(CAMERAS)
written = 0
start = time.time()

for round_idx in range(FRAMES_PER_CAMERA):
    for cam in CAMERAS:
        src = rng.choice(_LOCAL_PATHS)
        with open(src, "rb") as f:
            payload = f.read()
        ts_ms = int(time.time() * 1000)
        dest = f"{INBOX}/{cam}/frame_{ts_ms}.jpg"
        # Write directly to the UC volume path (POSIX-style under /Volumes).
        with open(dest, "wb") as f:
            f.write(payload)
        written += 1
    elapsed = time.time() - start
    LOG.info(
        "Round %d/%d done (%d/%d frames written, %.1fs elapsed)",
        round_idx + 1, FRAMES_PER_CAMERA, written, total, elapsed,
    )
    if round_idx < FRAMES_PER_CAMERA - 1:
        time.sleep(CADENCE_S)

LOG.info("Simulator done: wrote %d frames in %.1fs.", written, time.time() - start)